# FPGA-Accelerated Speaker Recognition on NBFM Signals via FINN

This notebook implements an end-to-end pipeline for deploying a CNN-based speaker recognition
model on an FPGA using the **FINN** framework by Xilinx Research Labs.

## Pipeline Overview

1. **Dataset Preparation** — Load pre-generated MFCC splits from `MFCC_datasets/` (produced by `MFCC_dataset_generation.ipynb`)
2. **Float Baseline Model** — Train a standard PyTorch CNN as a reference
3. **Quantization-Aware Training (QAT)** — Train Brevitas-quantized models at 16-bit, 8-bit, and 4-bit
4. **ONNX Export** — Export all models to FINN-compatible ONNX
5. **FINN Synthesis** — Run the FINN compiler pipeline targeting the **xck26-sfvc784-2LV-c** @ 200 MHz with Vitis HLS

### Target FPGA
- **Part:** `xck26-sfvc784-2LV-c` (Kria KV260 / K26 SOM)
- **Clock:** 200 MHz (5 ns period)
- **HLS Backend:** Vitis HLS

### Prerequisites
Run `MFCC_dataset_generation.ipynb` first to produce the MFCC splits in `MFCC_datasets/`:
- `MFCC_datasets/train_batches/`, `validation_batches/`, `test_batches/`
- `MFCC_datasets/element_spec.pkl`
- `MFCC_datasets/metadata.json`

### Changes from the Original Notebook
- **Framework migrated** from TensorFlow/Keras to PyTorch + Brevitas (required by FINN)
- **Dataset loaded from pre-generated MFCC splits** — no IQ files accessed here; raw extraction moved to `MFCC_dataset_generation.ipynb`
- **Model caching** — cells detect previously trained models and skip retraining to prevent overfitting
- **Early stopping** uses `patience=15` with best-model restoration (not appended to an already-trained model)
- **Removed** the `save_history` append-on-rerun pattern that accumulated duplicate epochs
- **All comments translated** to English
- **Batch Normalization folded** before export (FINN requirement)
- **Dropout removed** at export time (only active during training)

---
## 0. Environment Setup and Imports

In [4]:
# ============================================================
# 0. Environment Setup
# ============================================================
# This cell installs/verifies all dependencies.
# Run inside the FINN Docker container for full synthesis support.
# Outside the container, training and ONNX export still work.
# ============================================================

import subprocess, sys, importlib, os

def ensure_package(pkg_name, pip_name=None):
    """Install a package if not already available."""
    try:
        importlib.import_module(pkg_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name or pkg_name, '-q'])

ensure_package('torch')
ensure_package('brevitas')
ensure_package('librosa')
ensure_package('sklearn', 'scikit-learn')
ensure_package('onnx')
ensure_package('tqdm')
ensure_package('seaborn')

print('All base packages available.')

# Check for FINN availability (only inside FINN Docker)
FINN_AVAILABLE = False
try:
    import finn
    import finn.builder.build_dataflow as build
    import finn.builder.build_dataflow_config as build_cfg
    FINN_AVAILABLE = True
    print(f'FINN framework detected: {finn.__file__}')
except ImportError:
    print('WARNING: FINN framework not found. Training and ONNX export will work, '
          'but FPGA synthesis requires the FINN Docker container.')

All base packages available.
FINN framework detected: None


In [5]:
# ============================================================
# Core imports
# ============================================================

import numpy as np
import os
import json
import pickle
import copy
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import brevitas.nn as qnn
from brevitas.quant import Int8ActPerTensorFloat, Int8WeightPerTensorFloat
from brevitas.export import export_qonnx

# Brevitas only ships Int8 quantizers out of the box.
# For other bit-widths, the documented approach is to subclass
# the Int8 quantizer and override the bit_width attribute.

class Int4WeightPerTensorFloat(Int8WeightPerTensorFloat):
    """4-bit signed integer weight quantizer."""
    bit_width = 4

class Int16WeightPerTensorFloat(Int8WeightPerTensorFloat):
    """16-bit signed integer weight quantizer."""
    bit_width = 16

class Int16ActPerTensorFloat(Int8ActPerTensorFloat):
    """16-bit signed integer activation quantizer."""
    bit_width = 16

import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.auto import tqdm

# Reproducibility
RANDOM_SEED = 55
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cpu


In [9]:
# ============================================================
# Project directory structure
# ============================================================

BASE_DIR          = Path('.')                      # Project root
MFCC_DATASETS_DIR = BASE_DIR / 'MFCC_datasets'          # Pre-generated MFCC splits (from MFCC_dataset_generation.ipynb)
MODELS_DIR        = BASE_DIR / 'trained_models'    # Saved model weights
ONNX_DIR          = BASE_DIR / 'onnx_exports'      # Exported ONNX files
FINN_BUILD_DIR    = BASE_DIR / 'finn_build'        # FINN synthesis output
HISTORY_DIR       = BASE_DIR / 'training_history'  # Training histories

for d in [MODELS_DIR, ONNX_DIR, FINN_BUILD_DIR, HISTORY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directory structure ready.')

Directory structure ready.


---
## 1. Dataset Preparation — Load Pre-generated MFCC Splits

Loads MFCC features and split metadata produced by `MFCC_dataset_generation.ipynb`.
No raw IQ files are accessed here.

In [10]:
# ============================================================
# 1a. Load metadata from pre-generated MFCC dataset
# ============================================================
# Reads parameters saved by MFCC_dataset_generation.ipynb.
# No IQ files are accessed in this notebook.
# ============================================================

META_PATH = MFCC_DATASETS_DIR / 'metadata.json'

with open(META_PATH, 'r') as f:
    meta = json.load(f)

NUM_CLASSES      = meta['num_classes']
MFCC_INPUT_SHAPE = tuple(meta['MFCC_input_shape'])  # (N_MFCC, MFCC_FRAMES, 2)
N_MFCC           = meta['N_MFCC']
MFCC_FRAMES      = meta['MFCC_FRAMES']
SAMPLING_FREQ    = meta['SAMPLING_FREQ']
subfolders       = meta['subfolders']
labels           = np.array(meta['labels'], dtype=np.int64)

print(f'Metadata loaded from: {META_PATH}')
print(f'  Classes          : {NUM_CLASSES}  {subfolders}')
print(f'  MFCC input shape : {MFCC_INPUT_SHAPE}   (N_MFCC, MFCC_FRAMES, I/Q channels)')
print(f'  N_MFCC           : {N_MFCC}')
print(f'  MFCC_FRAMES      : {MFCC_FRAMES}')
print(f'  Sampling freq    : {SAMPLING_FREQ} Hz')
print(f'  Total samples    : {len(labels)}')

FileNotFoundError: [Errno 2] No such file or directory: 'MFCC_datasets/metadata.json'

In [ ]:
# ============================================================
# 1b. Visualization parameters
# ============================================================
# N_FFT and HOP_LENGTH are only needed for the MFCC specshow
# plots in cell 1e. All model-critical parameters are already
# loaded from metadata.json in cell 1a.
# ============================================================

N_FFT      = 2048  # FFT window size used during MFCC generation
HOP_LENGTH = 512   # Hop length used during MFCC generation

print(f'Visualization parameters: N_FFT={N_FFT}, HOP_LENGTH={HOP_LENGTH}')
print(f'MFCC input shape for model: {MFCC_INPUT_SHAPE}')

In [ ]:
# ============================================================
# 1c. Load MFCC splits from MFCC_datasets/
# ============================================================
# Loads the TF snapshot datasets saved by MFCC_dataset_generation.ipynb
# and converts each split to NumPy arrays for use with PyTorch.
# ============================================================

import tensorflow as tf

SPEC_PATH = MFCC_DATASETS_DIR / 'element_spec.pkl'
with open(SPEC_PATH, 'rb') as f:
    specs = pickle.load(f)


def batched_tf_dataset_to_numpy(path, spec):
    """Load a batched TF snapshot dataset and return (X, y) as NumPy arrays."""
    ds = tf.data.Dataset.load(str(path), element_spec=spec)
    X_list, y_list = [], []
    for x_batch, y_batch in ds:
        X_list.append(x_batch.numpy())
        y_list.append(y_batch.numpy())
    return np.concatenate(X_list, axis=0), np.concatenate(y_list, axis=0)


print('Loading train split...')
X_train, y_train = batched_tf_dataset_to_numpy(
    MFCC_DATASETS_DIR / 'train_batches', specs['train'])

print('Loading validation split...')
X_val, y_val = batched_tf_dataset_to_numpy(
    MFCC_DATASETS_DIR / 'validation_batches', specs['validation'])

print('Loading test split...')
X_test, y_test = batched_tf_dataset_to_numpy(
    MFCC_DATASETS_DIR / 'test_batches', specs['test'])

y_train = y_train.astype(np.int64)
y_val   = y_val.astype(np.int64)
y_test  = y_test.astype(np.int64)

print(f'\nSplits loaded:')
print(f'  Train : X={X_train.shape}, y={y_train.shape}')
print(f'  Val   : X={X_val.shape},   y={y_val.shape}')
print(f'  Test  : X={X_test.shape},  y={y_test.shape}')

In [ ]:
# ============================================================
# 1d. Create PyTorch DataLoaders
# ============================================================
# FINN/Brevitas uses PyTorch, so we convert NumPy -> Tensors.
# Channel layout: (batch, channels, height, width) for Conv2d.
# Original shape: (N_MFCC, MFCC_FRAMES, 2) -> rearrange to (2, N_MFCC, MFCC_FRAMES)
# ============================================================

def numpy_to_loader(X, y, batch_size=32, shuffle=True):
    """Convert NumPy arrays to a PyTorch DataLoader with channel-first format."""
    # Rearrange from (B, H, W, C) to (B, C, H, W)
    X_t = torch.from_numpy(X).permute(0, 3, 1, 2).contiguous()
    y_t = torch.from_numpy(y).long()
    ds = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)

BATCH_SIZE = 32
train_loader = numpy_to_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
val_loader   = numpy_to_loader(X_val, y_val, BATCH_SIZE, shuffle=False)
test_loader  = numpy_to_loader(X_test, y_test, BATCH_SIZE, shuffle=False)

# Verify shapes
xb, yb = next(iter(train_loader))
print(f'Batch X shape: {xb.shape}  (batch, channels, mfcc_coefs, frames)')
print(f'Batch y shape: {yb.shape}')

In [ ]:
# ============================================================
# 1e. Visualize sample MFCCs
# ============================================================

fig, axs = plt.subplots(1, 2, figsize=(14, 4))
sample_idx = 0
sample_label = y_train[sample_idx]

for ch, ch_name in enumerate(['I-channel', 'Q-channel']):
    img = librosa.display.specshow(
        X_train[sample_idx, :, :, ch],
        x_axis='time', sr=SAMPLING_FREQ, hop_length=HOP_LENGTH, ax=axs[ch]
    )
    fig.colorbar(img, ax=axs[ch], format='%+2.0f')
    axs[ch].set_title(f'MFCC {ch_name} — Speaker {sample_label}')
    axs[ch].set_xlabel('Time (s)')
    axs[ch].set_ylabel('MFCC Coefficient')

plt.tight_layout()
plt.show()

---
## 2. Model Definitions

We define **four** model variants that share the same CNN architecture:

| Variant | Weights | Activations | Notes |
|---------|---------|-------------|-------|
| **Float** | FP32 | FP32 | Baseline reference |
| **QAT-16** | INT16 | INT16 | Minimal quantization loss |
| **QAT-8** | INT8 | INT8 | Good accuracy / resource tradeoff |
| **QAT-4** | INT4 | INT8 | Aggressive quantization for minimal FPGA resources |

### Architecture
```
Input (2, 20, MFCC_FRAMES)
  -> Conv2d(32, 3x3, same) -> BN -> ReLU -> MaxPool(2x2)
  -> Conv2d(64, 3x3, same) -> BN -> ReLU -> MaxPool(2x2)
  -> Flatten -> Linear(128) -> ReLU -> Dropout(0.5)
  -> Linear(NUM_CLASSES) -> LogSoftmax
```

### FINN-Specific Design Choices
- **Brevitas quantized layers** replace standard `nn.Conv2d` / `nn.Linear` in QAT variants
- **BatchNorm** is kept during training but **folded into convolutions** before ONNX export
- **Dropout** is only active during training (`model.eval()` disables it)
- Activations use **quantized ReLU** (`qnn.QuantReLU`) in QAT variants
- The final layer uses `QuantIdentity` (no activation quantization) so FINN handles the output

In [ ]:
# ============================================================
# 2a. Float baseline model (standard PyTorch)
# ============================================================

class SpeakerCNN_Float(nn.Module):
    """Standard FP32 CNN for speaker recognition on MFCC features."""

    def __init__(self, num_classes, input_channels=2, n_mfcc=20, n_frames=None):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        # Compute flattened size dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, input_channels, n_mfcc, n_frames)
            flat_size = self.features(dummy).view(1, -1).shape[1]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

print('Float model class defined.')

In [ ]:
# ============================================================
# 2b. Brevitas quantized model (parameterized bit-width)
# ============================================================
# FINN requires Brevitas quantized layers for QAT models.
# We parameterize the bit-width so the same class serves
# 16-bit, 8-bit, and 4-bit variants.
# ============================================================

from brevitas.quant import IntBias
from brevitas.quant_tensor import QuantTensor
from brevitas.inject.enum import ScalingImplType, RestrictValueType
import brevitas.config as brev_cfg
brev_cfg.IGNORE_MISSING_KEYS = True  # Allows loading float weights into quant model


def get_weight_quant(bit_width):
    """Return a Brevitas weight quantizer class for the given bit width."""
    if bit_width == 16:
        return Int16WeightPerTensorFloat
    elif bit_width == 8:
        return Int8WeightPerTensorFloat
    elif bit_width == 4:
        return Int4WeightPerTensorFloat
    else:
        raise ValueError(f'Unsupported weight bit-width: {bit_width}')


def get_act_quant(bit_width):
    """Return a Brevitas activation quantizer class for the given bit width."""
    if bit_width == 16:
        return Int16ActPerTensorFloat
    elif bit_width in (8, 4):  # 4-bit weights use 8-bit activations for stability
        return Int8ActPerTensorFloat
    else:
        raise ValueError(f'Unsupported activation bit-width: {bit_width}')


class SpeakerCNN_Quant(nn.Module):
    """Brevitas-quantized CNN for FINN deployment.

    Parameters
    ----------
    num_classes : int
    weight_bits : int  (4, 8, or 16)
    act_bits    : int  (8 or 16; 4-bit weights default to 8-bit activations)
    """

    def __init__(self, num_classes, weight_bits=8, act_bits=None,
                 input_channels=2, n_mfcc=20, n_frames=None):
        super().__init__()
        if act_bits is None:
            act_bits = 8 if weight_bits <= 8 else weight_bits

        wq = get_weight_quant(weight_bits)
        aq = get_act_quant(act_bits)
        self.weight_bits = weight_bits
        self.act_bits = act_bits

        # Input quantization (tells FINN the input precision)
        self.inp_quant = qnn.QuantIdentity(act_quant=aq, return_quant_tensor=True)

        self.conv1 = qnn.QuantConv2d(
            input_channels, 32, kernel_size=3, padding=1,
            weight_bit_width=weight_bits, weight_quant=wq, bias=True
        )
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = qnn.QuantReLU(act_quant=aq, return_quant_tensor=True)
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = qnn.QuantConv2d(
            32, 64, kernel_size=3, padding=1,
            weight_bit_width=weight_bits, weight_quant=wq, bias=True
        )
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = qnn.QuantReLU(act_quant=aq, return_quant_tensor=True)
        self.pool2 = nn.MaxPool2d(2)

        self.flatten = nn.Flatten()

        # Compute flattened size
        with torch.no_grad():
            dummy = torch.zeros(1, input_channels, n_mfcc, n_frames)
            x = self.pool1(self.bn1(self.conv1(dummy)))
            x = self.pool2(self.bn2(self.conv2(x)))
            flat_size = x.view(1, -1).shape[1]

        self.fc1 = qnn.QuantLinear(
            flat_size, 128, bias=True,
            weight_bit_width=weight_bits, weight_quant=wq
        )
        self.relu3 = qnn.QuantReLU(act_quant=aq, return_quant_tensor=True)
        self.dropout = nn.Dropout(0.5)

        self.fc2 = qnn.QuantLinear(
            128, num_classes, bias=True,
            weight_bit_width=weight_bits, weight_quant=wq
        )

    def forward(self, x):
        x = self.inp_quant(x)
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.flatten(x)
        x = self.dropout(self.relu3(self.fc1(x)))
        x = self.fc2(x)
        return x

print('Quantized model class defined.')

---
## 3. Training Infrastructure

### Overfitting Prevention
- **Early stopping** monitors `val_loss` with `patience=15` and restores best weights
- **Model caching** — if a `.pt` checkpoint exists, the cell loads it and skips training
- The original notebook's `save_history` appended epochs on re-run, inflating curves — this is fixed
- **No redundant callbacks** that would cause double-saving or interfere with early stopping

In [ ]:
# ============================================================
# 3. Training and evaluation utilities
# ============================================================

class EarlyStopping:
    """Early stopping that saves the best model weights in memory."""
    def __init__(self, patience=15, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.best_state = None
        self.should_stop = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)
            print(f'  Restored best weights (val_loss={self.best_loss:.4f})')


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += xb.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        running_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += xb.size(0)
    return running_loss / total, correct / total


def train_model(model, name, train_loader, val_loader, device,
                epochs=100, lr=1e-4, patience=15):
    """
    Full training loop with early stopping and caching.

    If a checkpoint for `name` already exists, the model is loaded
    from disk and training is skipped entirely.

    Returns
    -------
    history : dict  with keys 'train_loss', 'val_loss', 'train_acc', 'val_acc'
    """
    ckpt_path = MODELS_DIR / f'{name}.pt'
    hist_path = HISTORY_DIR / f'{name}_history.json'

    # ---- Cache check: skip training if already done ----
    if ckpt_path.exists():
        print(f'[{name}] Checkpoint found at {ckpt_path} — loading and skipping training.')
        state = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(state, strict=False)
        model.to(device)
        if hist_path.exists():
            with open(hist_path) as f:
                return json.load(f)
        return {}

    # ---- Training ----
    print(f'[{name}] Training for up to {epochs} epochs (patience={patience})...')
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=7)
    es = EarlyStopping(patience=patience)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl, va = evaluate(model, val_loader, criterion, device)
        scheduler.step(vl)
        es.step(vl, model)

        history['train_loss'].append(tl)
        history['val_loss'].append(vl)
        history['train_acc'].append(ta)
        history['val_acc'].append(va)

        elapsed = time.time() - t0
        if epoch % 5 == 0 or epoch == 1 or es.should_stop:
            print(f'  Epoch {epoch:3d}/{epochs}  '
                  f'train_loss={tl:.4f}  val_loss={vl:.4f}  '
                  f'train_acc={ta:.3f}  val_acc={va:.3f}  '
                  f'[{elapsed:.1f}s]')
        if es.should_stop:
            print(f'  Early stopping at epoch {epoch}.')
            break

    es.restore(model)

    # Save checkpoint and history
    torch.save(model.state_dict(), ckpt_path)
    with open(hist_path, 'w') as f:
        json.dump(history, f)
    print(f'  Saved: {ckpt_path}')

    return history


def plot_history(history, title='Training History'):
    """Plot training/validation loss and accuracy."""
    if not history:
        print('No history to plot.')
        return
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    ax1.plot(history['train_loss'], label='Train')
    ax1.plot(history['val_loss'], label='Validation')
    ax1.set_title('Loss')
    ax1.set_xlabel('Epoch')
    ax1.legend()

    ax2.plot(history['train_acc'], label='Train')
    ax2.plot(history['val_acc'], label='Validation')
    ax2.set_title('Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylim([0, 1])
    ax2.legend()

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def full_evaluation(model, loader, device, class_names=None, title='Confusion Matrix'):
    """Run evaluation and show confusion matrix + classification report."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            preds = model(xb).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(yb.numpy())

    cm = confusion_matrix(all_labels, all_preds)
    if class_names is None:
        class_names = [str(i) for i in range(cm.shape[0])]

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(title)
    plt.tight_layout()
    plt.show()

    print(classification_report(all_labels, all_preds, target_names=class_names))
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return acc

print('Training utilities ready.')

---
## 4. Train All Model Variants

In [ ]:
# ============================================================
# 4a. Float baseline
# ============================================================

model_float = SpeakerCNN_Float(
    num_classes=NUM_CLASSES, input_channels=2,
    n_mfcc=N_MFCC, n_frames=MFCC_FRAMES
)
print(model_float)
print(f'Parameters: {sum(p.numel() for p in model_float.parameters()):,}')

hist_float = train_model(model_float, 'float_baseline',
                         train_loader, val_loader, DEVICE,
                         epochs=100, lr=1e-4, patience=15)
plot_history(hist_float, 'Float Baseline')

In [ ]:
# ============================================================
# 4b. QAT 16-bit
# ============================================================

model_q16 = SpeakerCNN_Quant(
    num_classes=NUM_CLASSES, weight_bits=16, act_bits=16,
    input_channels=2, n_mfcc=N_MFCC, n_frames=MFCC_FRAMES
)
print(f'QAT-16 parameters: {sum(p.numel() for p in model_q16.parameters()):,}')

hist_q16 = train_model(model_q16, 'qat_16bit',
                       train_loader, val_loader, DEVICE,
                       epochs=100, lr=1e-4, patience=15)
plot_history(hist_q16, 'QAT 16-bit')

In [ ]:
# ============================================================
# 4c. QAT 8-bit
# ============================================================

model_q8 = SpeakerCNN_Quant(
    num_classes=NUM_CLASSES, weight_bits=8, act_bits=8,
    input_channels=2, n_mfcc=N_MFCC, n_frames=MFCC_FRAMES
)
print(f'QAT-8 parameters: {sum(p.numel() for p in model_q8.parameters()):,}')

hist_q8 = train_model(model_q8, 'qat_8bit',
                      train_loader, val_loader, DEVICE,
                      epochs=100, lr=1e-4, patience=15)
plot_history(hist_q8, 'QAT 8-bit')

In [ ]:
# ============================================================
# 4d. QAT 4-bit (weights=4, activations=8)
# ============================================================

model_q4 = SpeakerCNN_Quant(
    num_classes=NUM_CLASSES, weight_bits=4, act_bits=8,
    input_channels=2, n_mfcc=N_MFCC, n_frames=MFCC_FRAMES
)
print(f'QAT-4 parameters: {sum(p.numel() for p in model_q4.parameters()):,}')

hist_q4 = train_model(model_q4, 'qat_4bit',
                      train_loader, val_loader, DEVICE,
                      epochs=120, lr=1e-4, patience=20)
plot_history(hist_q4, 'QAT 4-bit')

---
## 5. Evaluate All Models on Test Set

In [ ]:
# ============================================================
# 5. Test-set evaluation for all four variants
# ============================================================
# `subfolders` is already loaded from metadata.json in cell 1a.

class_names = subfolders if subfolders else None

results = {}
for name, model in [('Float', model_float), ('QAT-16', model_q16),
                     ('QAT-8', model_q8), ('QAT-4', model_q4)]:
    print(f'\n{"="*50}')
    print(f'  {name} Model — Test Set Evaluation')
    print(f'{"="*50}')
    model.to(DEVICE)
    acc = full_evaluation(model, test_loader, DEVICE,
                          class_names=class_names, title=f'{name} Confusion Matrix')
    results[name] = acc

print('\n\n=== Accuracy Summary ===')
for name, acc in results.items():
    print(f'  {name:10s}: {acc*100:.2f}%')

---
## 6. ONNX Export for FINN

FINN requires QONNX-format ONNX models exported via `brevitas.export.export_qonnx`.
Before export we:
1. Set the model to `eval()` mode (disables Dropout)
2. BatchNorm is automatically folded by Brevitas during export
3. Create a representative input tensor matching the expected shape

In [ ]:
# ============================================================
# 6a. Export float model to standard ONNX (for reference)
# ============================================================

import onnx

float_onnx_path = ONNX_DIR / 'speaker_cnn_float.onnx'

if not float_onnx_path.exists():
    model_float.eval().cpu()
    dummy_input = torch.randn(1, 2, N_MFCC, MFCC_FRAMES)
    torch.onnx.export(
        model_float, dummy_input, str(float_onnx_path),
        input_names=['input'], output_names=['output'],
        dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
        opset_version=13
    )
    print(f'Float ONNX saved: {float_onnx_path}')
else:
    print(f'Float ONNX already exists: {float_onnx_path}')

# Quick sanity check
onnx_model = onnx.load(str(float_onnx_path))
onnx.checker.check_model(onnx_model)
print('Float ONNX model is valid.')

In [ ]:
# ============================================================
# 6b. Export quantized models to QONNX (FINN-compatible)
# ============================================================

def export_brevitas_model(model, name, onnx_dir, n_mfcc, n_frames):
    """Export a Brevitas model to QONNX format for FINN."""
    out_path = onnx_dir / f'speaker_cnn_{name}.onnx'
    if out_path.exists():
        print(f'  [{name}] QONNX already exists: {out_path}')
        return out_path

    model.eval().cpu()
    dummy_input = torch.randn(1, 2, n_mfcc, n_frames)
    export_qonnx(model, dummy_input, str(out_path))
    print(f'  [{name}] QONNX exported: {out_path}')

    # Validate
    m = onnx.load(str(out_path))
    onnx.checker.check_model(m)
    print(f'  [{name}] ONNX validation passed.')
    return out_path


qonnx_paths = {}
for tag, mdl in [('qat_16bit', model_q16), ('qat_8bit', model_q8), ('qat_4bit', model_q4)]:
    qonnx_paths[tag] = export_brevitas_model(mdl, tag, ONNX_DIR, N_MFCC, MFCC_FRAMES)

print('\nAll QONNX exports complete.')
for k, v in qonnx_paths.items():
    print(f'  {k}: {v}')

---
## 7. FINN Synthesis Pipeline

This section runs the **FINN dataflow builder** to compile the QONNX models into
bitstream-ready IP cores for the target FPGA.

### Target Configuration
- **FPGA Part:** `xck26-sfvc784-2LV-c`
- **Clock Period:** 5.0 ns (200 MHz)
- **HLS Backend:** Vitis HLS
- **Board:** Kria KV260 SOM

### Build Steps (automated by FINN)
1. `step_qonnx_to_finn` — Convert QONNX to FINN-ONNX internal representation
2. `step_tidy_up` — Streamline, fold constants, remove unused nodes
3. `step_streamline` — Apply arithmetic optimizations, absorb BN into weights
4. `step_convert_to_hw` — Map layers to FINN HW operator library (HLS or RTL)
5. `step_create_dataflow_partition` — Partition into a single dataflow accelerator
6. `step_specialize_layers` — Select optimal compute engine for each layer
7. `step_target_fps_parallelization` — Set per-layer parallelism (PE/SIMD) for target throughput
8. `step_apply_folding_config` — Apply user-specified folding if provided
9. `step_minimize_bit_width` — Reduce internal bit-widths where safe
10. `step_generate_estimate_reports` — Resource and performance estimates
11. `step_hw_codegen` — Generate Vitis HLS C++ for each node
12. `step_hw_ipgen` — Run Vitis HLS to produce IP cores
13. `step_set_fifo_depths` — Size inter-layer FIFOs
14. `step_create_stitched_ip` — Stitch IP cores into a single AXI-Stream accelerator
15. `step_measure_rtlsim_performance` — RTL simulation to verify throughput
16. `step_out_of_context_synthesis` — Run Vivado OOC synthesis for resource usage
17. `step_synthesize_bitfile` — **(Optional)** Generate full bitstream with Vivado

> **Note:** Steps 11-17 require the FINN Docker container with Vitis HLS and Vivado installed.
> Outside the container, the pipeline stops after step 10 (estimates only).

In [ ]:
# ============================================================
# 7a. FINN build configuration
# ============================================================

# Target FPGA part and clock
FPGA_PART    = 'xck26-sfvc784-2LV-c'
CLOCK_NS     = 5.0        # 200 MHz
TARGET_FPS   = 10000      # Target throughput (inferences/sec)
BOARD        = 'KV260_SOM'

if FINN_AVAILABLE:
    from finn.builder.build_dataflow_config import (
        DataflowBuildConfig,
        ShellFlowType,
        VerificationStepType,
    )
    from finn.builder.build_dataflow_steps import (
        step_qonnx_to_finn,
        step_tidy_up,
        step_streamline,
        step_convert_to_hw,
        step_create_dataflow_partition,
        step_specialize_layers,
        step_target_fps_parallelization,
        step_apply_folding_config,
        step_minimize_bit_width,
        step_generate_estimate_reports,
        step_hw_codegen,
        step_hw_ipgen,
        step_set_fifo_depths,
        step_create_stitched_ip,
        step_measure_rtlsim_performance,
        step_out_of_context_synthesis,
        step_synthesize_bitfile,
    )

    # Define the full pipeline for bitstream-ready IP
    BUILD_STEPS = [
        step_qonnx_to_finn,
        step_tidy_up,
        step_streamline,
        step_convert_to_hw,
        step_create_dataflow_partition,
        step_specialize_layers,
        step_target_fps_parallelization,
        step_apply_folding_config,
        step_minimize_bit_width,
        step_generate_estimate_reports,
        step_hw_codegen,
        step_hw_ipgen,
        step_set_fifo_depths,
        step_create_stitched_ip,
        step_measure_rtlsim_performance,
        step_out_of_context_synthesis,
        step_synthesize_bitfile,
    ]

    print(f'FINN build steps configured: {len(BUILD_STEPS)} steps')
    print(f'Target: {FPGA_PART} @ {1000/CLOCK_NS:.0f} MHz')
else:
    print('FINN not available — synthesis cells will be skipped.')
    print('Run this notebook inside the FINN Docker container for full synthesis.')

In [ ]:
# ============================================================
# 7b. Run FINN synthesis for each QAT model
# ============================================================
# Each model gets its own build directory under finn_build/.
# The build is idempotent — if the output directory already
# contains a completed build, it is skipped.
# ============================================================

if FINN_AVAILABLE:
    for tag, onnx_path in qonnx_paths.items():
        build_dir = FINN_BUILD_DIR / tag
        bitfile_dir = build_dir / 'bitfile'

        # Skip if already synthesized
        if bitfile_dir.exists() and any(bitfile_dir.glob('*.bit')):
            print(f'[{tag}] Bitstream already exists in {bitfile_dir} — skipping.')
            continue

        print(f'\n{"="*60}')
        print(f'  FINN Build: {tag}')
        print(f'  Input ONNX: {onnx_path}')
        print(f'  Output dir: {build_dir}')
        print(f'{"="*60}\n')

        cfg = DataflowBuildConfig(
            output_dir           = str(build_dir),
            synth_clk_period_ns  = CLOCK_NS,
            fpga_part            = FPGA_PART,
            target_fps           = TARGET_FPS,
            board                = BOARD,
            shell_flow_type      = ShellFlowType.VITIS_ALVEO,
            # Use Vitis HLS as the HLS backend
            hls_clk_period_ns    = CLOCK_NS,
            vitis_platform       = '',
            # Generate IP, stitched design, and bitstream
            generate_outputs     = [
                build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
                build_cfg.DataflowOutputType.STITCHED_IP,
                build_cfg.DataflowOutputType.OOC_SYNTH,
                build_cfg.DataflowOutputType.BITFILE,
                build_cfg.DataflowOutputType.DEPLOYMENT_PACKAGE,
            ],
            steps                = BUILD_STEPS,
            # Verification at key stages
            verify_steps         = [
                VerificationStepType.QONNX_TO_FINN_PYTHON,
                VerificationStepType.STREAMLINED_PYTHON,
                VerificationStepType.TIDY_UP_PYTHON,
            ],
            # Save intermediate models for debugging
            save_intermediate_models = True,
        )

        # Launch the build
        t0 = time.time()
        build.build_dataflow_cfg(str(onnx_path), cfg)
        elapsed = time.time() - t0
        print(f'\n[{tag}] Build completed in {elapsed/60:.1f} minutes.')
else:
    print('FINN not available — skipping synthesis.')
    print('To run synthesis, start the FINN Docker container and re-run this notebook.')

In [ ]:
# ============================================================
# 7c. Display synthesis reports
# ============================================================

if FINN_AVAILABLE:
    import glob

    for tag in qonnx_paths:
        report_dir = FINN_BUILD_DIR / tag / 'report'
        if not report_dir.exists():
            print(f'[{tag}] No reports found.')
            continue

        print(f'\n{"="*60}')
        print(f'  Reports for: {tag}')
        print(f'{"="*60}')

        for rpt_file in sorted(report_dir.glob('*.json')):
            print(f'\n--- {rpt_file.name} ---')
            with open(rpt_file) as f:
                rpt = json.load(f)
            for k, v in rpt.items():
                print(f'  {k}: {v}')
else:
    print('FINN not available — no reports to display.')

---
## 8. Summary and Deployment Notes

### Build Outputs

After a successful FINN build, the following artifacts are generated per model variant:

| Directory | Contents |
|-----------|----------|
| `finn_build/<variant>/report/` | Resource estimates (LUT, BRAM, DSP, FF) and throughput |
| `finn_build/<variant>/stitched_ip/` | Vivado IP core ready for integration |
| `finn_build/<variant>/bitfile/` | Bitstream `.bit` + hardware handoff `.hwh` |
| `finn_build/<variant>/deploy/` | Deployment package with driver and bitstream |

### Deploying on the Kria KV260

1. Copy the `deploy/` folder to the KV260
2. Use PYNQ or the FINN Python runtime to load the accelerator
3. Feed MFCC features (shape `1×2×20×FRAMES`) to the accelerator input
4. Read the classification output from the accelerator

In [ ]:
# ============================================================
# 8. Final summary table
# ============================================================

print('\n' + '='*70)
print('  SPEAKER RECOGNITION CNN — MODEL COMPARISON')
print('='*70)
print(f'{"Model":<12} {"Test Acc":>10} {"Weight Bits":>12} {"Act Bits":>10} {"ONNX File":>30}')
print('-'*70)

rows = [
    ('Float',  results.get('Float',0),  'FP32', 'FP32', 'speaker_cnn_float.onnx'),
    ('QAT-16', results.get('QAT-16',0), '16',   '16',   'speaker_cnn_qat_16bit.onnx'),
    ('QAT-8',  results.get('QAT-8',0),  '8',    '8',    'speaker_cnn_qat_8bit.onnx'),
    ('QAT-4',  results.get('QAT-4',0),  '4',    '8',    'speaker_cnn_qat_4bit.onnx'),
]
for name, acc, wb, ab, fname in rows:
    print(f'{name:<12} {acc*100:>9.2f}% {wb:>12} {ab:>10} {fname:>30}')

print('-'*70)
print(f'Target FPGA: {FPGA_PART}  |  Clock: {1000/CLOCK_NS:.0f} MHz  |  HLS: Vitis HLS')
print('='*70)

In [ ]:
# ============================================================
# List all generated output files
# ============================================================

print('\nGenerated outputs:')
for d in [MFCC_DATASETS_DIR, MODELS_DIR, ONNX_DIR, HISTORY_DIR, FINN_BUILD_DIR]:
    print(f'\n  {d}/')
    if d.exists():
        for f in sorted(d.rglob('*')):
            if f.is_file():
                size_mb = f.stat().st_size / 1e6
                rel = f.relative_to(d)
                print(f'    {rel}  ({size_mb:.2f} MB)')